# OmniVoice — Robust Long-form Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binhminhanh1235/OmniVoice/blob/master/notebooks/OmniVoice.ipynb)

This notebook runs the patched fork:

- Repository: `binhminhanh1235/OmniVoice`
- Branch: `master`

The branch includes the upstream PR #259 silence/edge fix plus a robust long-form pipeline for reducing clipped phonemes, repeated phrases, skipped words, and sentence mixing.


## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [ ]:
!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@master"

## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!omnivoice-demo --share

## 3. Option B — Python API

### 3.1 Load Model

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    # Load a smaller English Whisper verifier on CPU.
    # This keeps T4 VRAM focused on OmniVoice.
    load_asr=True,
    asr_model_name="openai/whisper-small.en",
    asr_device="cpu",
)

print("Sampling rate:", model.sampling_rate)

## Robust long-form mode

Use this mode for long narration where exact wording matters.

Pipeline:

`semantic chunking → generate → Whisper verify → retry failed chunk → recursively split persistent failures → explicit-silence stitch`

Key safeguards:
- paragraph/sentence boundaries are preferred over commas;
- one `VoiceClonePrompt` is reused for all chunks;
- nested long-form splitting is disabled;
- chunk edge fades are disabled;
- Whisper checks omissions, repetition, word-count drift, and missing negations such as `not`, `no`, `never`, `without`.


In [ ]:
from omnivoice import (
    OmniVoiceGenerationConfig,
    RobustLongFormConfig,
    RobustLongFormGenerator,
)

robust_config = RobustLongFormConfig(
    max_chunk_words=24,
    max_chunk_chars=220,
    max_retries=3,
    max_split_depth=2,
    verify_with_asr=True,
    asr_model_name="openai/whisper-small.en",
    asr_device="cpu",
    max_wer=0.18,
    min_similarity=0.82,
    min_word_ratio=0.74,
    max_word_ratio=1.30,
    pause_ms=320,
    paragraph_pause_ms=460,
    strict=False,
)

robust_generator = RobustLongFormGenerator(
    model,
    robust_config,
)

generation_config = OmniVoiceGenerationConfig(
    num_step=32,
    guidance_scale=2.0,
    position_temperature=1.0,
    class_temperature=0.0,
    # RobustLongFormGenerator already owns the chunking.
    audio_chunk_threshold=1e9,
    # Preserve low-level phonemes at chunk boundaries.
    pad_duration=0.0,
    fade_duration=0.0,
    # PR #259 edge/silence controls.
    postprocess_output=True,
    output_min_silence_ms=650,
    output_keep_silence_ms=180,
    output_lead_silence_ms=80,
    output_trail_silence_ms=130,
    output_target_lead_silence_ms=0,
    output_target_trail_silence_ms=0,
)

print("Robust long-form pipeline ready.")

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [ ]:
from google.colab import files

print("Upload a clean 3-10 second reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

# Recommended: paste the exact transcript of the reference clip.
# Leave empty to let the CPU Whisper model transcribe it automatically.
REF_TEXT = ""

voice_prompt = model.create_voice_clone_prompt(
    ref_audio=ref_audio_path,
    ref_text=REF_TEXT.strip() or None,
    preprocess_prompt=True,
)

print("Reference text used by OmniVoice:")
print(voice_prompt.ref_text)

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    language="en",
    voice_clone_prompt=voice_prompt,
)

sf.write("clone_out.wav", audio[0], model.sampling_rate)
display(Audio(audio[0], rate=model.sampling_rate))

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/binhminhanh1235/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], model.sampling_rate)
display(Audio(audio[0], rate=model.sampling_rate))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], model.sampling_rate)
display(Audio(audio[0], rate=model.sampling_rate))

## Generate long-form narration

Requirements:
- `TEXT`: target narration text
- `voice_prompt`: reusable `VoiceClonePrompt`

The cell writes both WAV output and a CSV verification report.


In [ ]:
# Replace TEXT with your narration.
TEXT = """Let's be clear from the beginning.

This is not about refusing kindness to someone who is sick, grieving, poor, overwhelmed, or genuinely trying to rebuild their life.

This is not about becoming suspicious of everyone who needs help.

And this is not about calling people "toxic" because they disappointed you.

This is about repeated patterns.

Patterns that reject truth.
Patterns that avoid responsibility.
Patterns that turn compassion into permission.

Love is not unlimited access.

Forgiveness is not instant trust.

And helping a person is not the same as helping the pattern that keeps hurting them, others, and sometimes you.

So as we walk through these five patterns, do not use them to judge someone quickly. Use them first to examine the kind of help you are giving.
""".strip()

result = robust_generator.generate(
    TEXT,
    language="en",
    voice_clone_prompt=voice_prompt,
    generation_config=generation_config,
)

import soundfile as sf
from IPython.display import Audio, display
import pandas as pd

OUTPUT_PATH = "/content/omnivoice_robust_output.wav"
REPORT_PATH = "/content/omnivoice_robust_report.csv"

sf.write(OUTPUT_PATH, result.audio, result.sampling_rate)

report_df = pd.DataFrame(
    [
        {
            "chunk": i,
            "accepted": r.accepted,
            "attempts": r.attempts,
            "depth": r.depth,
            "wer": round(r.wer, 4),
            "similarity": round(r.similarity, 4),
            "word_ratio": round(r.word_ratio, 4),
            "critical_missing": ", ".join(r.critical_missing),
            "extra_repetitions": ", ".join(r.extra_repetitions),
            "expected": r.text,
            "whisper_transcript": r.transcript,
        }
        for i, r in enumerate(result.reports, 1)
    ]
)

report_df.to_csv(REPORT_PATH, index=False)

print("All chunks verified:", result.all_verified)
print("Final chunks:", len(result.chunks))
print("WAV:", OUTPUT_PATH)
print("Report:", REPORT_PATH)

display(report_df)
display(Audio(result.audio, rate=result.sampling_rate))